# 🔎 Hidden Hop - FC-MH


## 1. Install

In [ ]:
!pip install -q rank_bm25 openai tiktoken datasets sentence-transformers transformers torch numpy pandas

## 2. API key

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # reads .env into os.environ

In [ ]:
import os, getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")
print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

## 3. Imports

In [ ]:
import os
import re, time, json, string
import numpy as np
import pandas as pd
import tiktoken
from openai import OpenAI

# Provider: OpenAI by default. Set USE_OPENROUTER=1 (or leave OPENAI_API_KEY unset) to
# route through OpenRouter instead
if os.getenv("USE_OPENROUTER") == "1" or not os.getenv("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"],
                    base_url="https://openrouter.ai/api/v1")
    _pfx = lambda m: m if "/" in str(m) else "openai/" + str(m)
    _c0, _e0 = client.chat.completions.create, client.embeddings.create
    def _c(*a, **kw):
        if "model" in kw: kw["model"] = _pfx(kw["model"])
        return _c0(*a, **kw)
    def _e(*a, **kw):
        if "model" in kw: kw["model"] = _pfx(kw["model"])
        return _e0(*a, **kw)
    client.chat.completions.create, client.embeddings.create = _c, _e
    print("provider: OpenRouter")
else:
    client = OpenAI()
    print("provider: OpenAI")
tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")
print("Imports ready.")

## 4. Config

In [ ]:
# ---------------- CENTRAL CONFIG ----------------
DATASET        = "fcmh"          # "fcmh" or "fcsh"
RETRIEVER_NAME = "openai-large"          # bm25 | openai-small | openai-large | contriever | qwen3
TOP_K          = 3            # facts fed to the reader
READER_MODEL   = "gpt-4o-mini"   # NB: letter o, not 4.0 - "gpt-4.0-mini" 404s
READER_MODE    = "serial"        # "serial" = tell reader higher serial# = newer ; "plain" = no rule
MAX_FACTS      = None            # cap corpus for cheap testing; None = all
MAX_QUESTIONS  = None            # cap questions for cheap testing; None = all
DECAY_LAMBDA = 0.5    # sweep: 0.0 (baseline), 0.5, 1.0, 2.0, 5.0
MAP_IDS = False   # True: renumber 1..k, hide serials, decay-order. False: original serial mode.
REASONING_EFFORT = "medium"   # gpt-5.x only; None for non-reasoning models
SORT_BY = "serial" # "score" (decay/similarity order) or "serial" (ascending serial)

# qwen3 only:
QWEN_MODEL     = "Qwen/Qwen3-Embedding-4B"   # 8GB fp16. Fallback for small GPUs: "Qwen/Qwen3-Embedding-0.6B"
EMBED_BATCH    = 32              # lower this if Qwen3-4B OOMs on an 8GB card

# auto-detect GPU
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("No CUDA GPU detected -> running on CPU")
print("Config set:", RETRIEVER_NAME, "/", DATASET, "/ reader =", READER_MODE)

In [ ]:
import re as _rec

REASONING_RESERVE = 4096      # reasoning tokens are charged against the completion budget

def _is_reasoning_model(m):
    return bool(_rec.match(r"^(gpt-5|o\d)", str(m)))

if not getattr(client.chat.completions, "_compat_wrapped", False):
    _orig_create = client.chat.completions.create

    def _compat_create(*a, **kw):
        if _is_reasoning_model(kw.get("model", "")):
            if "max_tokens" in kw:                       # renamed on gpt-5.x
                kw["max_completion_tokens"] = kw.pop("max_tokens") + REASONING_RESERVE
            # temperature=0 is only rejected when reasoning_effort is set; with reasoning
            # off, keep temperature so the run stays deterministic like the gpt-4o-mini arm
            if REASONING_EFFORT:
                kw.setdefault("reasoning_effort", REASONING_EFFORT)
                kw.pop("temperature", None)
        return _orig_create(*a, **kw)

    client.chat.completions.create = _compat_create
    client.chat.completions._compat_wrapped = True

print("reader compat active |", READER_MODEL,
      "| reasoning:", _is_reasoning_model(READER_MODEL), REASONING_EFFORT)
print("Every cell below now works on gpt-4o-mini and gpt-5.x without edits.")


## 5. Load FC split

In [ ]:
from datasets import load_dataset

ds  = load_dataset("ai-hyz/MemoryAgentBench", split="Conflict_Resolution")
SRC = "factconsolidation_mh_262k" if DATASET == "fcmh" else "factconsolidation_sh_262k"
row = [x for x in ds if (x.get("metadata") or {}).get("source", "") == SRC][0]

context   = row["context"]
questions = row["questions"]
answers   = row["answers"]

print(f"Dataset   : {DATASET}  ({SRC})")
print(f"Context   : {len(context):,} chars (~{len(context)//4:,} tokens)")
print(f"Questions : {len(questions)}")
print(f"Q0: {questions[0][:100]}")
print(f"A0: {answers[0]}")

## 6. Parse the context into individual numbered facts
Each fact `"N. ..."` becomes one retrievable document, keeping its serial number (the recency signal for conflict resolution).

In [ ]:
def parse_facts(text):
    # Split right before each "N. " marker (same pattern as the Mem0 chunker).
    parts = re.split(r"(?=\b\d+\. )", text)
    parts = [p.strip() for p in parts if p.strip()]
    out = []
    for p in parts:
        m = re.match(r"^(\d+)\.\s", p)
        out.append({"serial": int(m.group(1)) if m else -1, "text": p})
    return out

# Alt line-based parser if facts look fragmented above:
# def parse_facts(text):
#     facts, cur = [], None
#     for line in text.split("\n"):
#         m = re.match(r"^\s*(\d+)\.\s", line)
#         if m:
#             if cur: facts.append(cur)
#             cur = {"serial": int(m.group(1)), "text": line.strip()}
#         elif cur:
#             cur["text"] += " " + line.strip()
#     if cur: facts.append(cur)
#     return facts

facts = parse_facts(context)
if MAX_FACTS:
    facts = facts[:MAX_FACTS]
docs = [f["text"] for f in facts]

SERIALS      = np.array([f["serial"] for f in facts])   # numpy array, doc-aligned
MAX_SERIAL   = SERIALS.max()

print(f"Parsed {len(facts)} numbered facts.")
print("First:", docs[0][:120])
print("Last :", docs[-1][:120])
print(SERIALS)

## 7. Retrievers

In [ ]:
class BM25Retriever:
    name = "bm25"
    def build(self, docs):
        from rank_bm25 import BM25Okapi
        self.docs = docs
        self.bm25 = BM25Okapi([d.lower().split() for d in docs])
    def retrieve(self, query, k):
        similarity = self.bm25.get_scores(query.lower().split())
        if DECAY_LAMBDA:
            age    = (MAX_SERIAL - SERIALS) / MAX_SERIAL
            scores = similarity * np.exp(-DECAY_LAMBDA * age)
        else:
            scores = similarity
        idx = np.argsort(scores)[::-1][:k]
        hits = [(facts[int(i)], float(scores[i])) for i in idx]   # currently in score order
        if SORT_BY == "serial":
            hits.sort(key=lambda x: x[0]["serial"], reverse=MAP_IDS)               # ascending serial (newest last)
        # SORT_BY == "score": leave as-is (best score first)
        return hits

class DenseRetriever:
    """Holds an L2-normalized doc matrix; cosine top-k."""
    name = "dense"
    def _encode_docs(self, docs):  raise NotImplementedError
    def _encode_query(self, query): raise NotImplementedError
    def build(self, docs):
        self.docs = docs
        M = np.asarray(self._encode_docs(docs), dtype="float32")
        M /= (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12)
        self.M = M
    def retrieve(self, query, k):
        q = np.asarray(self._encode_query(query), dtype="float32").ravel()
        q /= (np.linalg.norm(q) + 1e-12)
        similarity = self.M @ q
        if DECAY_LAMBDA:
            age    = (MAX_SERIAL - SERIALS) / MAX_SERIAL
            scores = similarity * np.exp(-DECAY_LAMBDA * age)
        else:
            scores = similarity
        idx = np.argsort(scores)[::-1][:k]
        # hits = [(facts[int(i)], float(scores[i])) for i in idx]
        # if not MAP_IDS:
        #     hits.sort(key=lambda x: x[0]["serial"])   # serial mode: show in serial order
        # return hits
        hits = [(facts[int(i)], float(scores[i])) for i in idx]   # currently in score order
        if SORT_BY == "serial":
            hits.sort(key=lambda x: x[0]["serial"], reverse=MAP_IDS)               # ascending serial (newest last)
        # SORT_BY == "score": leave as-is (best score first)
        return hits


class OpenAIDense(DenseRetriever):
    def __init__(self, model):
        self.model = model
        self.name = model
    def _embed(self, texts):
        out = []
        for i in range(0, len(texts), 256):
            resp = client.embeddings.create(model=self.model, input=texts[i:i+256])
            out.extend([d.embedding for d in resp.data])
            #print(f"  embedded {min(i+256, len(texts))}/{len(texts)}", end="\r")
        #print()
        return np.array(out)
    def _encode_docs(self, docs):  return self._embed(docs)
    def _encode_query(self, query): return self._embed([query])


class ContrieverDense(DenseRetriever):
    name = "contriever"
    def __init__(self, device="cpu", batch=64):
        import torch
        from transformers import AutoTokenizer, AutoModel
        self.torch, self.device, self.batch = torch, device, batch
        self.tok   = AutoTokenizer.from_pretrained("facebook/contriever")
        self.model = AutoModel.from_pretrained("facebook/contriever").to(device).eval()
    def _mean_pool(self, last_hidden, mask):
        mask = mask.unsqueeze(-1).float()
        return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    def _embed(self, texts):
        torch, vecs = self.torch, []
        for i in range(0, len(texts), self.batch):
            enc = self.tok(texts[i:i+self.batch], padding=True, truncation=True,
                           max_length=512, return_tensors="pt").to(self.device)
            with torch.no_grad():
                out = self.model(**enc)
            vecs.append(self._mean_pool(out.last_hidden_state, enc["attention_mask"]).cpu().numpy())
            #print(f"  encoded {min(i+self.batch, len(texts))}/{len(texts)}", end="\r")
        #print()
        return np.vstack(vecs)
    def _encode_docs(self, docs):  return self._embed(docs)
    def _encode_query(self, query): return self._embed([query])


class Qwen3Dense(DenseRetriever):
    def __init__(self, model_name, device="cpu", batch=32):
        from sentence_transformers import SentenceTransformer
        self.name  = model_name.split("/")[-1]
        self.batch = batch
        # fp16 on GPU halves VRAM (Qwen3-4B: ~8GB fp16 vs ~16GB fp32)
        model_kwargs = {"torch_dtype": "float16"} if device == "cuda" else {}
        self.model = SentenceTransformer(model_name, device=device, model_kwargs=model_kwargs)
        # Qwen3-Embedding expects an instruction on the query side only.
        self.q_instruct = "Instruct: Given a question, retrieve facts that help answer it\nQuery: "
    def _encode_docs(self, docs):
        return self.model.encode(docs, batch_size=self.batch,
                                 normalize_embeddings=False, show_progress_bar=True)
    def _encode_query(self, query):
        return self.model.encode([self.q_instruct + query], normalize_embeddings=False)


def make_retriever(name):
    if name == "bm25":         return BM25Retriever()
    if name == "openai-small": return OpenAIDense("text-embedding-3-small")
    if name == "openai-large": return OpenAIDense("text-embedding-3-large")
    if name == "contriever":   return ContrieverDense(device=DEVICE, batch=EMBED_BATCH)
    if name == "qwen3":        return Qwen3Dense(QWEN_MODEL, device=DEVICE, batch=max(8, EMBED_BATCH // 2))
    raise ValueError(f"unknown retriever: {name}")

print("Retrievers defined.")

## 8. Build the index for the selected retriever

In [ ]:
print(f"Building '{RETRIEVER_NAME}' index over {len(docs)} facts")
t0 = time.time()
retriever = make_retriever(RETRIEVER_NAME)
retriever.build(docs)
print(f"Index built in {time.time() - t0:.0f}s.")

## 9. Reader (identical to the Mem0 reader; only retrieval changed)
Retrieved facts are shown to the reader **sorted by serial number** so the recency rule is legible.

In [ ]:
hits = retriever.retrieve(questions[0], 10) 
hits

In [ ]:
SYSTEM_PLAIN = (
    "You are a knowledge management system. You will be given a Knowledge Pool of facts. "
    "Some facts may conflict. Answer the question using ONLY the Knowledge Pool. "
    "Give a very concise answer - one word or short phrase only."
)

# this is the one being used SYSTEM_SERIAL
SYSTEM_SERIAL = 'You are a knowledge management system. Answer the Question using ONLY the Knowledge Pool. All entities are synthetic or counterfactual, so your own world knowledge is WRONG here and must NEVER be used, even as a fallback. You may combine facts if needed. Your answer MUST be a word or phrase that literally appears inside the Knowledge Pool text. Each fact starts with a serial number; a HIGHER serial number means a MORE RECENT fact. Before you answer, check whether two facts give different values for the same relation of the same entity. If they do, answer with the one that has the HIGHER serial number - even when the lower-serial value is the one that matches the real world, which is the usual case. Give a very concise answer - one word or short phrase only.'

SYSTEM_MAPPED = (
    "You are a knowledge management system. You will be given a Knowledge Pool of facts. "
    "Some facts conflict - they refer to the same entity but with different values. "
    "Each fact starts with a number; a SMALLER number means a MORE RELEVANT fact. "
    "When facts conflict, always use the fact with the SMALLER number. "
    "Answer using ONLY the Knowledge Pool. Give a very concise answer - one word or short phrase only."
)

SYSTEM_SCORE = (
    "You are a knowledge management system. You will be given a Knowledge Pool of facts. "
    "Each fact starts with a score; a HIGHER score means a more relevant and more recent fact. "
    "Some facts conflict - they refer to the same entity but give different values. "
    "When facts conflict, prefer the fact with the HIGHER score. "
    "Answer using ONLY the Knowledge Pool. Give a very concise answer - one word or short phrase only."
)

def answer_question(question, k=TOP_K, verbose=False):
    hits = retriever.retrieve(question, k)        # already ordered per SORT_BY

    picked = [f for f, _ in hits]
    if MAP_IDS:
        lines = []
        for j, f in enumerate(picked):
            clean = re.sub(r"^\d+\.\s*", "", f["text"])   # strip leading "8530. "
            lines.append(f"{j+1}. {clean}")
        pool = "\n".join(lines)
        sys_prompt = SYSTEM_MAPPED
    else:
        if SORT_BY == "score":
            ranked = sorted(hits, key=lambda x: x[1], reverse=True)   # force descending score
            lines = []
            for f, s in ranked:
                clean = re.sub(r"^\d+\.\s*", "", f["text"])
                lines.append(f"score:{s:.3f} {clean}")
            pool = "\n".join(lines)
            sys_prompt = SYSTEM_SCORE
        else:
            # SORT_BY == "serial": keep the real serial in the text
            pool = "\n".join(f["text"] for f, _ in hits)
            sys_prompt = SYSTEM_SERIAL if READER_MODE == "serial" else SYSTEM_PLAIN

    user_msg = f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:"
    resp = client.chat.completions.create(
        model=READER_MODEL,
        messages=[{"role": "system", "content": sys_prompt},
                  {"role": "user",   "content": user_msg}],
        max_tokens=20, temperature=0.0, seed=42,
    )
    ans = resp.choices[0].message.content.strip()
    if verbose:
        print(sys_prompt + "\n")
        print(f"Q: {question[:120]}")
        print(pool)
        print(f"Answer: {ans}")
    return ans, len(picked)

# smoke test on Q0
a0, n0 = answer_question(questions[18], verbose=True)
print("Gold:", answers[18])

In [ ]:
import re

DECOMP_SYS = (
    "You decompose a multi-hop question into an ordered list of single-hop subquestions.\n"
    "Rules:\n"
    "- One subquestion per line, numbered '1.', '2.', ...\n"
    "- A later subquestion refers to the ANSWER of an earlier one with #k "
    "(e.g. #1 = answer to subquestion 1).\n"
    "- Each subquestion must be answerable on its own once the #k references are filled in.\n"
    "- If the question is already single-hop, output exactly one line.\n"
    "Output ONLY the numbered list, nothing else."
)

DECOMP_FEWSHOT = (
    "Question: What is the country of origin of the sport played by Christian Abbiati?\n"
    "1. What is the sport played by Christian Abbiati?\n"
    "2. What is the country of origin of #1?\n\n"
    "Question: Who is the spouse of the director of Inception?\n"
    "1. Who is the director of Inception?\n"
    "2. Who is the spouse of #1?\n\n"
    "Question: What is the capital of the country where the Eiffel Tower is located?\n"
    "1. In which country is the Eiffel Tower located?\n"
    "2. What is the capital of #1?\n"
)

def decompose_question(question):
    """Break a question into ordered subquestion templates (with #k placeholders).
    Returns (subqs, n_hops)."""
    user = DECOMP_FEWSHOT + "\nQuestion: " + question + "\n"
    resp = client.chat.completions.create(
        model=READER_MODEL,
        messages=[{"role": "system", "content": DECOMP_SYS},
                  {"role": "user",   "content": user}],
        max_tokens=256, temperature=0.0, seed=42,
    )
    text = resp.choices[0].message.content.strip()
    #print(text)
    subqs = []
    for line in text.split("\n"):
        m = re.match(r"^\s*\d+[.)]\s*(.+)$", line.strip())
        if m:
            subqs.append(m.group(1).strip())
    if not subqs:                 # fallback: treat as single-hop
        subqs = [question]
    return subqs, len(subqs)

def _fill(template, prev_answers):
    """Substitute #k with the k-th resolved answer (1-indexed)."""
    def repl(m):
        idx = int(m.group(1)) - 1
        return prev_answers[idx] if 0 <= idx < len(prev_answers) else m.group(0)
    return re.sub(r"#(\d+)", repl, template)

def answer_mh_question(question, k=TOP_K, verbose=False):
    subqs, n_hops = decompose_question(question)
    answers_so_far = []
    for i, tmpl in enumerate(subqs):
        filled = _fill(tmpl, answers_so_far)            # plug previous answers in
        ans, _ = answer_question(filled, k=k, verbose=verbose)           # reuse your single-shot solver
        answers_so_far.append(ans)
        if verbose:
            print(f"  [hop {i+1}/{n_hops}] {filled}")
            print(f"           -> {ans}")
    final = answers_so_far[-1] if answers_so_far else ""
    return final, n_hops

In [ ]:
print(questions[0])
pred, hops = answer_mh_question(questions[0], verbose=True)
print("Gold:", answers[0], "| pred:", pred, "| hops:", hops)

## 10. Run decompose arm - "Fixed" procedure in paper 

In [ ]:
def normalize(s):
    s = s.lower()
    s = "".join(c for c in s if c not in string.punctuation)
    return " ".join(s.split())

def score(pred, gts):
    if isinstance(gts, str): gts = [gts]
    if gts and isinstance(gts[0], list): gts = [g for sub in gts for g in sub]
    pred_n = normalize(pred)
    em  = any(normalize(g) == pred_n for g in gts)
    sem = any(normalize(g) in pred_n for g in gts)
    return em, sem

n_q  = min(MAX_QUESTIONS, len(questions)) if MAX_QUESTIONS else len(questions)
rows = []
t0   = time.time()
for qi in range(n_q):
    pred, n_ret = answer_mh_question(questions[qi], k=TOP_K)
    em, sem = score(pred, answers[qi])
    g = answers[qi]
    rows.append({"qi": qi, "prediction": pred,
                 "gold": g if isinstance(g, str) else " | ".join(map(str, g)),
                 "retrieved": n_ret, "EM": em, "SubEM": sem})
    print(f"Q{qi:3d}  pred={str(pred)[:24]:<24} gold={str(g)[:24]:<24} EM={em} SubEM={sem}")

df = pd.DataFrame(rows)
print(f"\nDone in {time.time() - t0:.0f}s")
print("=" * 55)
print(f"RESULTS  retriever={RETRIEVER_NAME}  dataset={DATASET}  reader={READER_MODE}")
print(f"  Exact Match     : {df['EM'].mean()*100:.1f}%")
print(f"  Substring Match : {df['SubEM'].mean()*100:.1f}%")
print(f"  Avg retrieved   : {df['retrieved'].mean():.1f}")

## 12. Single Retrive - no decomposition


In [ ]:
n_q = min(MAX_QUESTIONS, len(questions)) if MAX_QUESTIONS else len(questions)
rows_oner, t0 = [], time.time()
for qi in range(n_q):
    try:
        pred, n_ret = answer_question(questions[qi], k=TOP_K)
        err = ""
    except Exception as e:
        pred, n_ret, err = "", 0, f"{type(e).__name__}: {e}"[:140]
    em, sem = score(pred, answers[qi])
    g = answers[qi]
    rows_oner.append({"qi": qi, "prediction": pred,
                      "gold": g if isinstance(g, str) else " | ".join(map(str, g)),
                      "retrieved": n_ret, "llm_calls": 1,
                      "EM": em, "SubEM": sem, "error": err})
    print(f"Q{qi:3d}  pred={str(pred)[:24]:<24} gold={str(g)[:24]:<24} EM={em} SubEM={sem}"
          + (f"   !! {err}" if err else ""))

df_oner = pd.DataFrame(rows_oner)
n_err = int((df_oner["error"] != "").sum())
print()
print(f"Done in {time.time() - t0:.0f}s")
print("=" * 66)
print(f"SECTION 12 (OneR, no decomposition)  retriever={RETRIEVER_NAME}  dataset={DATASET}  "
      f"reader={READER_MODEL}  k={TOP_K}")
print(f"  Exact Match     : {df_oner['EM'].mean()*100:.1f}%")
print(f"  Substring Match : {df_oner['SubEM'].mean()*100:.1f}%")
print(f"  Avg retrieved   : {df_oner['retrieved'].mean():.1f}")
print(f"  Avg LLM calls   : 1.0")
if n_err:
    clean = df_oner[df_oner["error"] == ""]
    print(f"  !! {n_err} question(s) errored and are scored as wrong here. "
          f"Re-run them before quoting any number."
          + (f" Over the {len(clean)} clean rows: SubEM {clean['SubEM'].mean()*100:.1f}%."
             if len(clean) else ""))

fn = f"oner_{RETRIEVER_NAME}_{DATASET}_{READER_MODE}_k{TOP_K}_{READER_MODEL}.json"
with open(fn, "w", encoding="utf-8") as f:
    json.dump({"settings": {"method": "oner_no_decomposition", "dataset": DATASET,
                            "retriever": RETRIEVER_NAME, "reader_model": READER_MODEL,
                            "reader_mode": READER_MODE, "top_k": TOP_K,
                            "decay_lambda": DECAY_LAMBDA, "sort_by": SORT_BY,
                            "map_ids": MAP_IDS, "n_facts": len(facts), "n_q": n_q,
                            "n_errors": n_err},
               "scores": {"exact_match": float(df_oner["EM"].mean()*100),
                          "substring_match": float(df_oner["SubEM"].mean()*100),
                          "avg_llm_calls": 1.0},
               "results": df_oner.to_dict(orient="records")}, f, indent=2, default=str)
print("Saved:", fn)
df_oner.head(20)


## 13. Recursive decomposition - adaptive

In [ ]:
# ---------------- 13. config ----------------
MAX_SPLIT_DEPTH = 2        # how deep a subquestion may be split again. 0 disables splitting.
SOLVE_K         = TOP_K    # facts retrieved per subquestion
SOLVE_K_DEEP    = 0        # optional deeper pool, retried only on a node that could not answer
SOLVE_VERBOSE   = False    # print every node as it is solved
SOLVE_RETRIES   = 3        # retries for transient API errors

ANSWER_SYS = (
    "You are a knowledge management system. Answer the Question using ONLY the Knowledge Pool. "
    "All entities are synthetic or counterfactual, so your own world knowledge is WRONG here and "
    "must NEVER be used, even as a fallback. You may combine facts if needed. Your answer MUST be "
    "a word or phrase that literally appears inside the Knowledge Pool text. If the pool truly "
    "contains nothing related, output exactly: unknown. "
    "Each fact starts with a serial number; a HIGHER serial number means a MORE RECENT fact. "
    "Before you answer, check whether two facts give different values for the same relation of "
    "the same entity. If they do, answer with the one that has the HIGHER serial number - even "
    "when the lower-serial value is the one that matches the real world, which is the usual case. "
    "Give a very concise answer - one word or short phrase only."
)

# The same prompt with the refusal removed, for a node that has to commit.
GUESS_SYS = ANSWER_SYS.replace(
    "If the pool truly contains nothing "
    "related, output exactly: unknown. ",
    "Even when the pool is a poor match, give the closest word or phrase in it; never refuse and "
    "never answer 'unknown'. ")

# Reached only after ANSWER_SYS has said "unknown". Finds the entity that must be resolved first;
# never answers the question.
BRIDGE_SYS = (
    "You are a knowledge management system working ONLY over the given Knowledge Pool. "
    "All entities are synthetic or counterfactual, so your own world knowledge is WRONG here; "
    "never use it.\n"
    "A reader has already tried and failed to answer the Question from this pool. Do NOT answer "
    "the question. Your only job is to find the stepping stone.\n"
    "The pool often does not record the asked relation for the asked entity, but does record some "
    "OTHER relation of that entity, which names a NEW entity - and the asked relation is recorded "
    "for the new entity instead. The pool may not say what sport someone plays but may say what "
    "position they play; it may not give a spouse's citizenship but may name the spouse; it may "
    "not say who runs a product but may name the company that made it.\n"
    "Find the ONE fact in the pool that names that new entity, and copy it verbatim. The fact will "
    "normally mention an entity the question already names - that is how it connects. What makes "
    "it a stepping stone is the NEW name it introduces.\n"
    "A fact is NOT a stepping stone if it introduces no new name, or if it is about an entity the "
    "question is not asking about.\n"
    "Output EXACTLY ONE line:\n"
    "  BRIDGE: <one fact from the pool, copied verbatim>\n"
    "  NONE\n"
    "If no fact in the pool names such an entity, output exactly: NONE"
)

# Turns (question, bridge fact) into two subquestions, the second chaining on #1.
SPLIT_SYS = (
    "A question cannot be answered directly from the Knowledge Pool, but a bridge fact was found. "
    "Split the question into EXACTLY TWO simpler subquestions using that bridge fact.\n"
    "All entities are synthetic or counterfactual; never use your own world knowledge to decide "
    "what to ask.\n"
    "Rules:\n"
    "- Subquestion 1 must be answerable by the bridge fact.\n"
    "- Subquestion 2 must contain the placeholder #1 (the answer of subquestion 1) and, once #1 "
    "is filled in, must answer the original question.\n"
    "- Both subquestions must be about entities named in the question or the bridge fact you were "
    "given. Never mention an entity from the examples.\n"
    "- Output EXACTLY two lines, nothing else:\n"
    "1. <subquestion 1>\n"
    "2. <subquestion 2 containing #1>"
)

# Examples are invented entities, verified absent from the corpus and the question set by
# fc_prompts_fixed.verify_clean(). Corpus entities here leak into real answers.
SPLIT_FEWSHOT = (
    "### EXAMPLES (format only - do NOT answer these, do NOT reuse their entities)\n"
    "Question: In which city was the author of The Ashen Ledger born?\n"
    "Bridge fact: 4102. The author of The Ashen Ledger is Ilvane Sabreth.\n"
    "1. Who is the author of The Ashen Ledger?\n"
    "2. In which city was #1 born?\n\n"
    "Question: Which country is the spouse of Marren Volkov a citizen of?\n"
    "Bridge fact: 7731. Marren Volkov is married to Orrin Fennwick.\n"
    "1. Who is Marren Volkov married to?\n"
    "2. Which country is #1 a citizen of?\n"
    "### END EXAMPLES\n\n"
    "Now the REAL question. Both subquestions must be about the entities in THIS question and its "
    "bridge fact, and nothing from the examples above.\n"
)

print(f"Section 13 config: max_split_depth={MAX_SPLIT_DEPTH} k={SOLVE_K} reader={READER_MODEL}")


In [ ]:
import re, time

def _chat(system, user, max_tokens=96):
    """One reader call. Transient API errors are retried so they cannot become wrong answers."""
    for attempt in range(SOLVE_RETRIES):
        try:
            r = client.chat.completions.create(
                model=READER_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user",   "content": user}],
                max_tokens=max_tokens, temperature=0.0, seed=42,
            )
            return r.choices[0].message.content.strip()
        except Exception:
            if attempt == SOLVE_RETRIES - 1:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("SOLVE_RETRIES must be >= 1")


def _norm(s):
    return " ".join(re.sub(r"[^a-z0-9\s]", "", str(s).lower()).split())


def retrieve_pool(question, k=None):
    """Retrieve facts for one subquestion, oldest first.

    The flow chart's 'conflict resolve' is the serial rule, applied in the prompt rather than by a
    filter here: the pool is serial-ascending and ANSWER_SYS says the higher serial wins.
    """
    hits = retriever.retrieve(question, SOLVE_K if k is None else k)
    hits = sorted(hits, key=lambda h: h[0]["serial"])
    return "\n".join(f["text"] for f, _ in hits), hits


def answer_from_pool(question, pool):
    """Answer from the pool, or 'unknown' if it does not contain the answer.

    This is the flow chart's check(): the decision and the answer are one call, and 'unknown' is
    the refusal that fires the split.
    """
    raw = _chat(ANSWER_SYS, f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:",
                max_tokens=20)
    return raw.split("\n")[0].strip(), raw


def guess_from_pool(question, pool):
    """Commit an answer for a node nothing could repair: the closest thing in the pool rather
    than 'unknown'."""
    raw = _chat(GUESS_SYS, f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nAnswer:",
                max_tokens=20)
    return raw.split("\n")[0].strip(), raw


def find_bridge(question, pool):
    """Name the entity that has to be resolved first. Returns (bridge_fact, raw) or (None, raw).

    Reached only after the pool has failed to answer, and it never answers the question itself, so
    it cannot overrule the reader. Its job is the evidence the split is conditioned on.
    """
    raw = _chat(BRIDGE_SYS, f"[Knowledge Pool]\n{pool}\n\nQuestion: {question}\nOutput:")
    line = next((l.strip() for l in raw.split("\n") if l.strip()), "")
    m = re.match(r"BRIDGE\s*:\s*(.+)$", line, re.I)
    if not m or _norm(m.group(1)) in ("", "none"):
        return None, raw
    return m.group(1).strip(), raw


def split(question, bridge):
    """(question, bridge fact) -> ([subq1, subq2-containing-#1], raw), or (None, raw) when the
    split is unusable: unparseable, no #1 to chain on, or subq1 just restates the parent."""
    raw = _chat(SPLIT_SYS,
                SPLIT_FEWSHOT + f"\nQuestion: {question}\nBridge fact: {bridge}\n",
                max_tokens=128)
    subqs = []
    for line in raw.split("\n"):
        m = re.match(r"^\s*[12][.)]\s*(.+)$", line.strip())
        if m:
            subqs.append(m.group(1).strip())
    if len(subqs) < 2 or "#1" not in subqs[1] or _norm(subqs[0]) == _norm(question):
        return None, raw
    return subqs[:2], raw


def _show(node):
    """Print one node: its pool, the raw LLM output at each step, and what it decided."""
    pad = "   " * node["depth"]
    print(f"{pad}[d{node['depth']}] {node['question']}")
    for p in node["pool"]:
        print(f"{pad}      [s={p['serial']:>6}] {p['text'][:100]}")
    for key in ("answer_raw", "deep_raw", "bridge_raw", "split_raw", "guess_raw"):
        if node.get(key):
            print(f"{pad}      {key:<10} {node[key][:110]!r}")
    if node["decision"] == "split":
        print(f"{pad}      bridge     {node['bridge'][:100]}")
        print(f"{pad}      -> split into: 1. {node['split'][0]}  |  2. {node['split'][1]}")
    else:
        print(f"{pad}      -> {node['decision']}: {node['answer']!r}")


def solve(question, depth, run):
    """One node of the flow chart: retrieve, try to answer, and split only if the pool cannot."""
    # A split can regenerate a subquestion an earlier hop already solved. Cached per question.
    key = _norm(question)
    if key in run["memo"]:
        cached = run["memo"][key]
        run["trace"].append({"depth": depth, "question": question, "pool": cached["pool"],
                             "decision": "reused (already solved this hop)",
                             "answer": cached["answer"]})
        run["reused"] += 1
        return cached["answer"]

    pool, hits = retrieve_pool(question)
    node = {"depth": depth, "question": question,
            "pool": [{"serial": f["serial"], "score": s, "text": f["text"]} for f, s in hits]}
    run["trace"].append(node)
    run["max_depth"] = max(run["max_depth"], depth)

    def leaf(decision, ans):
        node.update(decision=decision, answer=ans)
        run["memo"][key] = {"answer": ans, "pool": node["pool"]}
        if run["verbose"]:
            _show(node)
        return ans

    def give_up(decision):
        """No answer and no repair left: guess rather than return 'unknown'."""
        g, node["guess_raw"] = guess_from_pool(question, pool)
        run["calls"] += 1
        run["guesses"] += 1
        return leaf(decision, g)

    # No split budget left, so a refusal could not be acted on: commit in one call.
    if depth >= MAX_SPLIT_DEPTH:
        g, node["guess_raw"] = guess_from_pool(question, pool)
        run["calls"] += 1
        run["guesses"] += 1
        return leaf("guessed (depth cap)", g)

    ans, node["answer_raw"] = answer_from_pool(question, pool)
    run["calls"] += 1
    if _norm(ans) != "unknown":
        return leaf("answered", ans)

    if SOLVE_K_DEEP > SOLVE_K:
        deep_pool, deep_hits = retrieve_pool(question, k=SOLVE_K_DEEP)
        ans, node["deep_raw"] = answer_from_pool(question, deep_pool)
        run["calls"] += 1
        run["deepened"] += 1
        node["deep_pool"] = [{"serial": f["serial"], "score": sc, "text": f["text"]}
                             for f, sc in deep_hits]
        if _norm(ans) != "unknown":
            return leaf("answered (deeper pool)", ans)
        pool = deep_pool          # the bridge and the guess get the wider pool too

    bridge, node["bridge_raw"] = find_bridge(question, pool)
    run["calls"] += 1
    if bridge is None:
        return give_up("guessed (no bridge)")

    pair, node["split_raw"] = split(question, bridge)
    run["calls"] += 1
    if pair is None:
        return give_up("guessed (split failed)")

    run["splits"] += 1
    node.update(decision="split", bridge=bridge, split=pair)
    if run["verbose"]:
        _show(node)
    node["answer"] = solve_sequence(pair, depth + 1, run)   # re-enters the loop, one level deeper
    run["memo"][key] = {"answer": node["answer"], "pool": node["pool"]}
    return node["answer"]


def solve_sequence(subqs, depth, run):
    """The for-loop. Answers each subquestion in order, substituting earlier answers into later
    #k placeholders; the sequence's answer is its last answer.

    No early exit on 'unknown': every hop is attempted, because an abandoned question scores the
    same as a wrong one and abandoning loses the hops that have not run yet.
    """
    resolved = []
    for tmpl in subqs:
        resolved.append(solve(_fill(tmpl, resolved), depth, run))
    return resolved[-1] if resolved else ""


def answer_question_recursive(question, verbose=None):
    """Entry point, drop-in for section 10's answer_mh_question. Returns (answer, run); run
    carries branches / calls / splits / max_depth / branch_answers / trace."""
    verbose = SOLVE_VERBOSE if verbose is None else verbose
    subqs, n_branch = decompose_question(question)
    run = {"branches": n_branch, "calls": 1, "splits": 0, "guesses": 0, "reused": 0,
           "deepened": 0, "max_depth": 0, "memo": {}, "trace": [], "verbose": verbose}
    if verbose:
        print(f"decompose -> {n_branch} subquestion(s):")
        for s in subqs:
            print("   -", s)
    final = solve_sequence(subqs, 0, run)
    run["branch_answers"] = [n["answer"] for n in run["trace"] if n["depth"] == 0]
    run.pop("verbose")
    run.pop("memo")
    return final, run


print("ready: answer_question_recursive(q)  |  solve / solve_sequence / find_bridge / split")

In [ ]:
# smoke test: one question, every node printed
_qi = 0
_pred, _run = answer_question_recursive(questions[_qi], verbose=True)

print(f"\nQ{_qi}: {questions[_qi]}")
print(f"gold : {answers[_qi]}")
print(f"pred : {_pred!r}")
print(f"branches={_run['branches']}  splits={_run['splits']}  guesses={_run['guesses']}  "
      f"max_depth={_run['max_depth']}  llm_calls={_run['calls']}")

In [ ]:
# ---------------- 13. run all questions & score ----------------
if "score" not in globals():          # section 10 not run in this session
    import string
    def normalize(s):
        s = "".join(c for c in str(s).lower() if c not in string.punctuation)
        return " ".join(s.split())
    def score(pred, gts):
        if isinstance(gts, str): gts = [gts]
        if gts and isinstance(gts[0], list): gts = [g for sub in gts for g in sub]
        p = normalize(pred)
        return any(normalize(g) == p for g in gts), any(normalize(g) in p for g in gts)


def run_all(depth=None, n=None, save=True, verbose_rows=True):
    """Score every question. `depth` temporarily overrides MAX_SPLIT_DEPTH, so a no-split run goes
    through this identical loop instead of a parallel copy of it."""
    global MAX_SPLIT_DEPTH
    keep = MAX_SPLIT_DEPTH
    if depth is not None:
        MAX_SPLIT_DEPTH = depth
    n_q = n or (min(MAX_QUESTIONS, len(questions)) if MAX_QUESTIONS else len(questions))
    rows, t0 = [], time.time()
    try:
        for qi in range(n_q):
            try:
                pred, run = answer_question_recursive(questions[qi])
                err = ""
            except Exception as e:      # still failing after SOLVE_RETRIES: flag, never hide
                pred, err = "", f"{type(e).__name__}: {e}"[:140]
                run = {"branches": 0, "calls": 0, "splits": 0, "guesses": 0, "max_depth": 0}
            em, sem = score(pred, answers[qi])
            g = answers[qi]
            rows.append({"qi": qi, "prediction": pred,
                         "gold": g if isinstance(g, str) else " | ".join(map(str, g)),
                         "branches": run["branches"], "splits": run["splits"],
                         "guesses": run["guesses"], "max_depth": run["max_depth"],
                         "llm_calls": run["calls"],
                         "EM": em, "SubEM": sem, "error": err})
            if verbose_rows:
                print(f"Q{qi:3d} pred={str(pred)[:22]:<22} gold={str(g)[:22]:<22} "
                      f"EM={em} SubEM={sem} d={run['max_depth']} "
                      f"splits={run['splits']} calls={run['calls']}"
                      + (f"   !! {err}" if err else ""))
    finally:
        MAX_SPLIT_DEPTH = keep

    df = pd.DataFrame(rows)
    d_used = keep if depth is None else depth
    print(f"\nDone in {time.time() - t0:.0f}s")
    print("=" * 66)
    print(f"SECTION 13  retriever={RETRIEVER_NAME}  dataset={DATASET}  reader={READER_MODEL}  "
          f"max_split_depth={d_used}  k={SOLVE_K}")
    print(f"  Exact Match     : {df['EM'].mean()*100:.1f}%")
    print(f"  Substring Match : {df['SubEM'].mean()*100:.1f}%")
    print(f"  Avg LLM calls   : {df['llm_calls'].mean():.1f}")
    print(f"  Split fired on  : {(df['splits'] > 0).mean()*100:.1f}% of questions")
    print(f"  Guessed a hop on: {(df['guesses'] > 0).mean()*100:.1f}% of questions "
          f"(the pool could not answer it and the split did not repair it)")
    n_err = int((df["error"] != "").sum())
    if n_err:
        clean = df[df["error"] == ""]
        msg = (f"  !! {n_err} question(s) errored and are scored as wrong here. "
               f"Re-run them before quoting any number.")
        if len(clean):
            msg += f" Over the {len(clean)} clean rows: SubEM {clean['SubEM'].mean()*100:.1f}%."
        print(msg)
        print("     " + "; ".join(f"Q{r.qi}: {r.error}" for r in df[df["error"] != ""].itertuples())[:300])

    if save:
        fn = (f"rec_{RETRIEVER_NAME}_{DATASET}_d{d_used}_k{SOLVE_K}"
              f"_lam{DECAY_LAMBDA}_{READER_MODEL}.json")
        with open(fn, "w", encoding="utf-8") as f:
            json.dump({"settings": {"method": "recursive_decomposition", "dataset": DATASET,
                                    "retriever": RETRIEVER_NAME, "reader_model": READER_MODEL,
                                    "max_split_depth": d_used, "retrieve_k": SOLVE_K,
                                    "decay_lambda": DECAY_LAMBDA, "sort_by": SORT_BY,
                                    "map_ids": MAP_IDS, "n_q": n_q, "n_errors": n_err},
                       "scores": {"exact_match": float(df["EM"].mean()*100),
                                  "substring_match": float(df["SubEM"].mean()*100),
                                  "avg_llm_calls": float(df["llm_calls"].mean())},
                       "results": df.to_dict(orient="records")}, f, indent=2, default=str)
        print("Saved:", fn)
    return df


df_rec = run_all()
df_rec.head(20)

### Inspect one question

In [ ]:
_qi = 0                      # any question index
_pred, _run = answer_question_recursive(questions[_qi], verbose=True)

print("\n--- compact trace ---")
for _n in _run["trace"]:
    print(f"d{_n['depth']}  {_n['decision']:<24} {_n['question'][:70]}")
    for _k, _label in (("answer_raw", "answer"), ("bridge_raw", "bridge"), ("split_raw", "split ")):
        if _n.get(_k):
            print(f"      {_label}: {_n[_k][:110]!r}")
    print(f"      -> {str(_n.get('answer'))[:80]!r}")
print(f"\npred: {_pred!r}   gold: {answers[_qi]}   "
      f"calls={_run['calls']} splits={_run['splits']} guesses={_run['guesses']} "
      f"max_depth={_run['max_depth']}")

## 14. Extras

In [ ]:
# ---------------- 14.0 helpers for the extra runs ----------------
import glob, threading
from concurrent.futures import ThreadPoolExecutor

HERE = os.path.abspath(os.getcwd())
if os.path.basename(HERE).lower() != "notebooks" and os.path.isdir(os.path.join(HERE, "notebooks")):
    HERE = os.path.join(HERE, "notebooks")
RUNS_DIR = os.path.join(HERE, "outputs", "fcmh")
CACHE_DIR = os.path.join(HERE, "cache", "fcmh")
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
RUNS = {}
_tl = threading.local()                  # what the current question did: calls, plans, retrievals, hops

# Record every retrieval, plan and reader call per question. Wrapped once; re-running is safe.
if not getattr(retriever, "_logged", False):
    _retrieve0 = retriever.retrieve
    def _logged_retrieve(query, k):
        hits = _retrieve0(query, k)
        if getattr(_tl, "retrievals", None) is not None:
            _tl.retrievals.append({"query": query, "k": k, "serials": [f["serial"] for f, _ in hits]})
        return hits
    retriever.retrieve, retriever._logged = _logged_retrieve, True

if not getattr(decompose_question, "_logged", False):
    _decompose0 = decompose_question
    def decompose_question(question):
        subqs, n = _decompose0(question)
        if getattr(_tl, "plans", None) is not None:
            _tl.plans.append(subqs)
        return subqs, n
    decompose_question._logged = True

if not getattr(answer_question, "_logged", False):
    _answer0 = answer_question
    def answer_question(question, k=None, verbose=False):
        ans, n = _answer0(question, k=TOP_K if k is None else k, verbose=verbose)
        if getattr(_tl, "hops", None) is not None:
            _tl.hops.append({"q": question, "a": ans})
        return ans, n
    answer_question._logged = True

if not getattr(client.chat.completions.create, "_counted", False):
    _create0 = client.chat.completions.create
    def _counted_create(*a, **kw):
        _tl.calls = getattr(_tl, "calls", 0) + 1
        return _create0(*a, **kw)
    _counted_create._counted = True
    client.chat.completions.create = _counted_create


def reader(system, user, max_tokens=20):
    """One reader call with this notebook's settings."""
    r = client.chat.completions.create(model=READER_MODEL, max_tokens=max_tokens, temperature=0.0, seed=42,
                                       messages=[{"role": "system", "content": system},
                                                 {"role": "user", "content": user}])
    return (r.choices[0].message.content or "").strip()


def run(name, fn, tag, workers=8):
    """fn(question) -> (prediction, info) over every question; saved as fcmh_<name>_<reader>_<tag>.json.
    An existing file is loaded instead."""
    path = os.path.join(RUNS_DIR, f"fcmh_{name}_{READER_MODEL}_{tag}.json")
    if os.path.exists(path):
        saved = json.load(open(path, encoding="utf-8"))
        print(f"loaded {os.path.basename(path)}: {saved['summary']}")
        return saved

    def one(qi):
        _tl.calls, _tl.plans, _tl.retrievals, _tl.hops = 0, [], [], []
        try:
            (pred, info), err = fn(questions[qi]), ""
        except Exception as e:
            pred, info, err = "", {}, f"{type(e).__name__}: {e}"[:200]
        em, sem = score(pred, answers[qi])
        return {"qi": qi, "question": questions[qi], "prediction": pred, "gold": " | ".join(map(str, answers[qi])),
                "EM": bool(em), "SubEM": bool(sem), "llm_calls": _tl.calls, "plans": _tl.plans,
                "retrievals": _tl.retrievals, "fixed_hops": _tl.hops, "error": err, **info}

    t0 = time.time()
    with ThreadPoolExecutor(workers) as ex:
        rows = list(ex.map(one, range(len(questions))))
    n = len(rows)
    summary = {"n": n, "SubEM": round(100 * sum(r["SubEM"] for r in rows) / n, 2),
               "EM": round(100 * sum(r["EM"] for r in rows) / n, 2),
               "calls_per_q": round(sum(r["llm_calls"] for r in rows) / n, 3),
               "n_errors": sum(bool(r["error"]) for r in rows), "wall_s": round(time.time() - t0)}
    if any("splits" in r for r in rows):
        summary["split_fired_pct"] = round(100 * sum(r.get("splits", 0) > 0 for r in rows) / n, 2)
        summary["refused_pct"] = round(100 * sum(bool(r.get("refused")) for r in rows) / n, 2)
    if any("retries" in r for r in rows):
        summary["retry_fired_pct"] = round(100 * sum(r.get("retries", 0) > 0 for r in rows) / n, 2)
        summary["refused_pct"] = round(100 * sum(bool(r.get("refused")) for r in rows) / n, 2)
    settings = {"run": name, "tag": tag, "reader_model": READER_MODEL, "reasoning_effort": REASONING_EFFORT,
                "retriever": RETRIEVER_NAME, "decay_lambda": DECAY_LAMBDA, "sort_by": SORT_BY,
                "solve_k": SOLVE_K, "max_split_depth": MAX_SPLIT_DEPTH}
    saved = {"settings": settings, "summary": summary, "rows": rows}
    json.dump(saved, open(path, "w", encoding="utf-8"), indent=1, ensure_ascii=False, default=str)
    print(f"saved {os.path.basename(path)}: {summary}")
    if summary["n_errors"]:
        print("  !! error rows score as wrong - run again with a new tag before quoting this run")
    return saved


# the three procedures of Table 1
def single(k):
    return lambda q: (answer_question(q, k=k)[0], {})


def fixed(k):
    return lambda q: (answer_mh_question(q, k=k)[0], {})


def adaptive(q):
    """Section 13 at the current SOLVE_K and MAX_SPLIT_DEPTH."""
    pred, r = answer_question_recursive(q)
    return pred, {"splits": r["splits"], "guesses": r["guesses"], "max_depth": r["max_depth"],
                  "branches": r["branches"], "trace": r["trace"],
                  "refused": any(_norm(n.get("answer_raw", "")) == "unknown" for n in r["trace"])}


print("extra runs are saved to", RUNS_DIR)

In [ ]:
# ---------------- 14.1 depth sweep: every procedure at k = 3 / 5 / 10 ----------------
# Fixed plan at k=3 and 5, and adaptive plan at k=10 (x3), also have older untraced runs in paper_results.
for k in (3, 5):
    RUNS[f"single_k{k}"] = run(f"single_k{k}", single(k), "sweep")
    RUNS[f"fixed_k{k}"] = run(f"fixed_k{k}", fixed(k), "sweep")

MAX_SPLIT_DEPTH = 2
for k, tag in ((5, "sweep"), (10, "r4")):
    SOLVE_K = k
    RUNS[f"adaptive_k{k}"] = run(f"adaptive_k{k}_d2", adaptive, tag)
SOLVE_K = TOP_K

In [ ]:
# ---------------- 14.2 the reader with no evidence ----------------
PARAMETRIC_SYS = ("Answer the question from your own knowledge of the world. "
                  "Give a very concise answer - one word or short phrase only.")

RUNS["closed"] = run("closed", lambda q: (reader(SYSTEM_SERIAL, f"[Knowledge Pool]\n\n\nQuestion: {q}\nAnswer:"), {}), "r1")
RUNS["parametric"] = run("parametric", lambda q: (reader(PARAMETRIC_SYS, f"Question: {q}\nAnswer:"), {}), "r1")

In [ ]:
# ---------------- 14.3 single query over a 512-token chunked index ----------------
enc = tiktoken.get_encoding("cl100k_base")
CHUNKS, cur, n_tok = [], [], 0
for f in facts:
    t = len(enc.encode(f["text"] + "\n"))
    if cur and n_tok + t > 512:
        CHUNKS.append(cur)
        cur, n_tok = [], 0
    cur.append(f)
    n_tok += t
CHUNKS.append(cur)

cached = [m for m in map(np.load, glob.glob(os.path.join(CACHE_DIR, "chunks512_openai-large_*.npy")))
          if m.shape[0] == len(CHUNKS)]
if cached:
    CHUNK_M = cached[0]
else:
    CHUNK_M = np.asarray(OpenAIDense("text-embedding-3-large")._embed(
        ["\n".join(f["text"] for f in c) for c in CHUNKS]), dtype="float32")
    CHUNK_M /= np.linalg.norm(CHUNK_M, axis=1, keepdims=True) + 1e-12
    np.save(os.path.join(CACHE_DIR, "chunks512_openai-large_notebook.npy"), CHUNK_M)
print(f"{len(CHUNKS)} chunks, {np.mean([len(c) for c in CHUNKS]):.1f} facts each on average")


def chunked(k):
    def fn(q):
        v = np.asarray(retriever._encode_query(q), dtype="float32").ravel()
        top = np.argsort(CHUNK_M @ (v / np.linalg.norm(v)))[::-1][:k]
        picked = sorted((CHUNKS[i] for i in top), key=lambda c: c[0]["serial"])
        pool = "\n".join(f["text"] for c in picked for f in c)
        return (reader(SYSTEM_SERIAL, f"[Knowledge Pool]\n{pool}\n\nQuestion: {q}\nAnswer:"),
                {"chunks": [int(i) for i in top], "facts_shown": sum(map(len, picked))})
    return fn


for k in (1, 3, 10):
    RUNS[f"chunked_k{k}"] = run(f"chunked_k{k}", chunked(k), "r1")

### 14.5 Decomposition + rewrite: rephrase and retry instead of the split

The control for §3.3 of the paper: a hidden hop is not fixed by rewriting the query. This is section 13's adaptive plan with one change. When a hop's reader returns `unknown`, the hop is **rephrased and retrieved again** instead of being split around a bridge fact. Everything else is the same: decomposer, Answer and Commit prompts, trigger, cap (`MAX_SPLIT_DEPTH` retries) and k=3. The rephrased query only changes what is retrieved; the reader still answers the hop's original question. If a hidden hop were a wording problem, this would recover as many questions as the split does.

In [ ]:
# 14.4 decomposition + rewrite: rephrase and retry instead of the split 
REPHRASE_SYS = (
    "A reader searched a knowledge base of one-line facts for the Question and could not find the answer in what "
    "came back. Rewrite the Question as a different search query asking for exactly the same information: reword "
    "the relation (synonyms, other phrasings of the same relation) and the way the entity is referred to. Keep the "
    "meaning identical. Do not answer the question, never use your own knowledge of the world, do not add any "
    "entity that is not in the Question, and do not repeat an earlier query. "
    "Output only the rewritten question, on one line.")


def rephrase(question, tried):
    """A new search query for the hop, or None if the reader gives nothing new."""
    earlier = "\n".join(f"- {t}" for t in tried)
    raw = _chat(REPHRASE_SYS, f"Question: {question}\nEarlier queries that failed:\n{earlier}\nRewritten question:",
                max_tokens=64)
    line = next((l.strip() for l in raw.split("\n") if l.strip()), "")
    line = re.sub(r"^(rewritten question|question)\s*:\s*", "", line, flags=re.I).strip()
    return (line if line and all(_norm(line) != _norm(t) for t in tried) else None), raw


def solve_retry(question, trace):
    """One hop: answer from its pool; on a refusal rephrase the query and retry; commit at the cap."""
    tried, query = [], question
    for depth in range(MAX_SPLIT_DEPTH + 1):
        pool, hits = retrieve_pool(query)
        node = {"depth": depth, "question": question, "query": query,
                "pool": [{"serial": f["serial"], "score": s, "text": f["text"]} for f, s in hits]}
        trace.append(node)
        tried.append(query)
        if depth >= MAX_SPLIT_DEPTH:                           # retries used up: commit, as section 13 does
            ans, node["guess_raw"] = guess_from_pool(question, pool)
            node.update(decision="guessed (retry cap)", answer=ans)
            return ans
        ans, node["answer_raw"] = answer_from_pool(question, pool)
        if _norm(ans) != "unknown":
            node.update(decision="answered", answer=ans)
            return ans
        new_query, node["rephrase_raw"] = rephrase(question, tried)
        if new_query is None:                                   # nothing new to try: commit
            ans, node["guess_raw"] = guess_from_pool(question, pool)
            node.update(decision="guessed (rephrase unusable)", answer=ans)
            return ans
        node.update(decision="rephrased", rewrite=new_query)
        query = new_query


def adaptive_retry(q):
    subqs, _ = decompose_question(q)
    trace, found = [], []
    for tmpl in subqs:
        found.append(solve_retry(_fill(tmpl, found), trace))
    return (found[-1] if found else ""), {
        "retries": sum(n.get("decision") == "rephrased" for n in trace), "trace": trace,
        "refused": any(_norm(n.get("answer_raw", "")) == "unknown" for n in trace)}


SOLVE_K, MAX_SPLIT_DEPTH = 3, 2
for tag in ("r1", "r2"):
    RUNS[f"retry_k3_{tag}"] = run("retry_k3_d2", adaptive_retry, tag)
SOLVE_K = TOP_K

In [ ]:
# 14.5b second reader: gpt-5.6-luna
assert REASONING_EFFORT, "gpt-5.6-luna needs REASONING_EFFORT (low / medium / high)"
_main_reader = READER_MODEL
READER_MODEL = "gpt-5.6-luna"
try:
    MAX_SPLIT_DEPTH = 2
    for k in (10, 5):
        RUNS[f"luna_single_k{k}"] = run(f"single_k{k}", single(k), "r1", workers=6)
        RUNS[f"luna_fixed_k{k}"] = run(f"fixed_k{k}", fixed(k), "r1", workers=6)
        SOLVE_K = k
        RUNS[f"luna_adaptive_k{k}"] = run(f"adaptive_k{k}_d2", adaptive, "r1", workers=6)
    SOLVE_K = 3                        # one more run (2026-09-17): adaptive only, at Table 1's k
    RUNS["luna_adaptive_k3"] = run("adaptive_k3_d2", adaptive, "r1", workers=6)
finally:
    READER_MODEL, SOLVE_K = _main_reader, TOP_K

In [ ]:
# 14.5c open-weight readers on DeepInfra: every Table 1 procedure at k = 3 and 5 
# Qwen3.5-35B-A3B and Gemma 4 26B-A4B through OpenRouter with thinking OFF
import random, time

OPEN_READERS = {"qwen3.5-35b-a3b": "qwen/qwen3.5-35b-a3b",
                "gemma-4-26b-a4b-it": "google/gemma-4-26b-a4b-it"}
OPEN_ROUTING = {"provider": {"order": ["DeepInfra"], "allow_fallbacks": False},
                "reasoning": {"enabled": False}}
assert os.getenv("USE_OPENROUTER") == "1", "these readers are served through OpenRouter only"

_create_before = client.chat.completions.create


OPEN_RETRIES = 8          # DeepInfra throttles in bursts ("temporarily rate-limited upstream") and there is
                          # no fallback: sections 9/10 have no retry, so an unretried 429 is a wrong answer


def _open_create(*a, **kw):
    if kw.get("model") not in OPEN_READERS:
        return _create_before(*a, **kw)
    kw["model"] = OPEN_READERS[kw["model"]]
    kw["extra_body"] = {**(kw.get("extra_body") or {}), **OPEN_ROUTING}
    for attempt in range(OPEN_RETRIES):
        try:
            return _create_before(*a, **kw)
        except Exception as e:
            if "429" not in str(e) or attempt == OPEN_RETRIES - 1:
                raise
            _tl.calls = max(getattr(_tl, "calls", 1) - 1, 0)   # a throttled attempt is not a reader call
            time.sleep(min(2 ** attempt, 30) + random.random())


_main_reader, _main_effort = READER_MODEL, REASONING_EFFORT
client.chat.completions.create = _open_create
try:
    REASONING_EFFORT, MAX_SPLIT_DEPTH = None, 2        # no reasoning for these readers; recorded as such
    for READER_MODEL in OPEN_READERS:
        for k in (3, 5):
            RUNS[f"{READER_MODEL}_single_k{k}"] = run(f"single_k{k}", single(k), "r1", workers=6)
            RUNS[f"{READER_MODEL}_fixed_k{k}"] = run(f"fixed_k{k}", fixed(k), "r1", workers=6)
            SOLVE_K = k
            RUNS[f"{READER_MODEL}_adaptive_k{k}"] = run(f"adaptive_k{k}_d2", adaptive, "r1", workers=6)
    # A run with error rows scores those questions as wrong and must not be quoted (see run()). Redo each such
    # run once under tag r2, with the 429 retry above and fewer workers. 2026-09-17: the first Gemma k=3 runs
    # had 21 (fixed) and 2 (adaptive) error rows, all DeepInfra 429s; the r1 files are kept as the record.
    for key in [x for x in RUNS if x.split("_")[0] in OPEN_READERS and RUNS[x]["summary"]["n_errors"]]:
        READER_MODEL, name = RUNS[key]["settings"]["reader_model"], RUNS[key]["settings"]["run"]
        k = int(re.search(r"_k(\d+)", name).group(1))
        SOLVE_K = k
        fn = single(k) if name.startswith("single") else fixed(k) if name.startswith("fixed") else adaptive
        RUNS[key] = run(name, fn, "r2", workers=3)
finally:
    client.chat.completions.create = _create_before
    READER_MODEL, REASONING_EFFORT, SOLVE_K = _main_reader, _main_effort, TOP_K

In [ ]:
# 14.6 everything run or loaded above 
pd.DataFrame([{"run": k, **v["summary"]} for k, v in RUNS.items()]).set_index("run")